In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/cluster_threshold_tables"
import os
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
import os
import pandas as pd

SAVE_DIR = "/content/drive/MyDrive/cluster_threshold_tables"

files = {
    "cluster_44_55": "cluster_44_55.csv",
    "cluster_59_4":  "cluster_59_4.csv",
    "cluster_74_25": "cluster_74_25.csv",
    "cluster_150":   "cluster_150.csv",
    "cluster_200":   "cluster_200.csv",
    "cluster_250":   "cluster_250.csv",
}

loaded = {}

for var, fname in files.items():
    path = os.path.join(SAVE_DIR, fname)
    if os.path.exists(path):
        loaded[var] = pd.read_csv(path)
        print(f"Loaded {var} | shape = {loaded[var].shape}")
    else:
        print(f"❌ Missing {fname}")

# assign variables
cluster_44_55 = loaded.get("cluster_44_55")
cluster_59_4  = loaded.get("cluster_59_4")
cluster_74_25 = loaded.get("cluster_74_25")
cluster_150   = loaded.get("cluster_150")
cluster_200   = loaded.get("cluster_200")
cluster_250   = loaded.get("cluster_250")

Installations

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, re
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from IPython.display import display

Paths & Parameters

In [ ]:
# -------------------
# Paths (EDIT IF NEEDED)
# -------------------
CRASHES_PATH       = "/content/drive/My Drive/Crashes_in_DC.csv"
CAMERAS_PATH       = "/content/drive/My Drive/Automated_Traffic_Enforcement.csv"
SPEEDHUMPS_PATH    = "/content/drive/My Drive/Speed_Humps.csv"
OUT_DIR            = "/content/drive/My Drive/outputs"
SPEED_GEOJSON_PATH = "/content/drive/My Drive/Roadway_SubBlock.geojson"
os.makedirs(OUT_DIR, exist_ok=True)

# -------------------
# Parameters
# -------------------
DATE_START = "2020-01-01"
DATE_END   = "2025-04-30"
MAR_MIN    = 100


Helper functions, gets rid of junk data and gives everything a standardized name.

In [ ]:
def clean_intersection(s):
    if s is None or (isinstance(s, float) and np.isnan(s)): return ""
    s = str(s).strip()
    if re.fullmatch(r"(nan|none|null|missing)", s, flags=re.I): return ""
    if "Intersecting RouteID" in s or "*" in s: return ""
    s = re.sub(r"\s*[/@&]\s*", " & ", s)
    return re.sub(r"\s+", " ", s).strip()

def extract_street_name(row):
    txt = None
    if "MAR_Address" in row and isinstance(row["MAR_Address"], str):
        txt = row["MAR_Address"]
    elif "ADDRESS" in row and isinstance(row["ADDRESS"], str):
        txt = row["ADDRESS"]
    if txt:
        t = txt.upper().strip()
        t = re.split(r"\s*&\s*|\s*@\s*|\s*/\s*|,| - ", t)[0]
        t = re.sub(r"\b(NE|NW|SE|SW)\b", "", t)
        return re.sub(r"\s+", " ", t).strip()
    inter = str(row.get("NEARESTINTSTREETNAME", "")).upper().strip()
    if inter:
        return re.split(r"\s*&\s*|\s*@\s*|\s*/\s*|,| - ", inter)[0].strip()
    return ""

def standardize_name(df, out_col="NAME", candidates=None, fallback_series=None):
    if candidates is None: candidates = []
    out = pd.Series("", index=df.index, dtype=object)
    for c in candidates:
        if c in df.columns:
            cand = df[c].fillna("").astype(str).str.strip()
            out = np.where(out != "", out, cand)
    if fallback_series is not None:
        fb = fallback_series.fillna("").astype(str).str.strip()
        out = np.where(out != "", out, fb)
    out = pd.Series(out, index=df.index).replace("", "(UNNAMED)")
    df[out_col] = out
    return df


Load and clean crashes. Filter for MAR score and for date.
Also define the calculations for the severity sum.

In [ ]:
# -------------------
# 4) Load & clean crashes
#    (WITH severity components; no distances yet)
# -------------------
df = pd.read_csv(CRASHES_PATH, dtype={"STREETSEGID": str}, low_memory=False)

# Keep speeding-related only
df["SPEEDING_INVOLVED"] = pd.to_numeric(df["SPEEDING_INVOLVED"], errors="coerce")
df = df[df["SPEEDING_INVOLVED"] > 0].copy()

# Valid lat/lon
df = df.dropna(subset=["LATITUDE", "LONGITUDE"]).copy()
df["LATITUDE"]  = pd.to_numeric(df["LATITUDE"], errors="coerce")
df["LONGITUDE"] = pd.to_numeric(df["LONGITUDE"], errors="coerce")
df = df.dropna(subset=["LATITUDE", "LONGITUDE"]).copy()

# Date window
df["FROMDATE"] = pd.to_datetime(df["FROMDATE"], errors="coerce")
df = df[(df["FROMDATE"] >= DATE_START) & (df["FROMDATE"] <= DATE_END)].copy()

# MAR quality (if present)
if "MAR_SCORE" in df.columns:
    df["MAR_SCORE"] = pd.to_numeric(df["MAR_SCORE"], errors="coerce")
    df = df[df["MAR_SCORE"] >= MAR_MIN].copy()

# ----- Injury columns & severity components -----
injury_categories = {
    "BICYCLIST":  ["MAJORINJURIES_BICYCLIST","MINORINJURIES_BICYCLIST","UNKNOWNINJURIES_BICYCLIST","FATAL_BICYCLIST"],
    "DRIVER":     ["MAJORINJURIES_DRIVER","MINORINJURIES_DRIVER","UNKNOWNINJURIES_DRIVER","FATAL_DRIVER"],
    "PEDESTRIAN": ["MAJORINJURIES_PEDESTRIAN","MINORINJURIES_PEDESTRIAN","UNKNOWNINJURIES_PEDESTRIAN","FATAL_PEDESTRIAN"],
    "PASSENGER":  ["MAJORINJURIESPASSENGER","MINORINJURIESPASSENGER","FATALPASSENGER"],
    "OTHER":      ["MAJORINJURIESOTHER","MINORINJURIESOTHER","FATALOTHER"],
}

fatal_cols = [c for cols in injury_categories.values() for c in cols if "FATAL" in c]
major_cols = [c for cols in injury_categories.values() for c in cols if "MAJOR" in c]
minor_cols = [c for cols in injury_categories.values() for c in cols if "MINOR" in c]

# Ensure all referenced injury columns exist & are numeric
for col in set(fatal_cols + major_cols + minor_cols):
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
    else:
        df[col] = 0

# Keep only crashes with at least one injury (exclude property-only)
injury_cols_all = fatal_cols + major_cols + minor_cols
df = df[df[injury_cols_all].sum(axis=1) > 0].copy()

# ----- Row-level injury totals -----
df["TOTAL_FATALITIES"] = df[fatal_cols].sum(axis=1)
df["TOTAL_MAJOR_INJURIES"] = df[major_cols].sum(axis=1)
df["TOTAL_MINOR_INJURIES"] = df[minor_cols].sum(axis=1)

# Weighted severity score (keep your 7-4-1 system)
df["SEVERITY_SCORE"] = (
    7 * df["TOTAL_FATALITIES"] +
    4 * df["TOTAL_MAJOR_INJURIES"] +
    1 * df["TOTAL_MINOR_INJURIES"]
)

df["CAPPED_SEVERITY"] = np.where(
    df["TOTAL_FATALITIES"] > 0, 7,
    np.where(
        df["TOTAL_MAJOR_INJURIES"] > 0, 4,
        np.where(df["TOTAL_MINOR_INJURIES"] > 0, 1, 0)
    )
)

# ----- Row-level flags needed for later cluster metrics -----
# Severe crash = any fatality or major injury
df["IS_SEVERE_CRASH"] = (
    (df["TOTAL_FATALITIES"] > 0) | (df["TOTAL_MAJOR_INJURIES"] > 0)
).astype(int)

# Fatal crash = any fatality
df["IS_FATAL_CRASH"] = (df["TOTAL_FATALITIES"] > 0).astype(int)

# Optional helper: count of serious outcomes
df["MAJOR_OR_FATAL_COUNT"] = (
    df["TOTAL_FATALITIES"] + df["TOTAL_MAJOR_INJURIES"]
)

# ----- Standardize names (no geometry ops yet) -----
if "NEARESTINTSTREETNAME" not in df.columns:
    df["NEARESTINTSTREETNAME"] = ""

df["NEARESTINTSTREETNAME"] = df["NEARESTINTSTREETNAME"].apply(clean_intersection)
df["PRIMARY_STREET"] = df.apply(extract_street_name, axis=1).str.upper()

df = standardize_name(
    df,
    out_col="NAME",
    candidates=["PRIMARY_STREET", "NEARESTINTSTREETNAME"],
    fallback_series=None
)

print(f"[Crashes after filters] {len(df):,} rows, with severity components computed.")

print(
    df[
        [
            "TOTAL_FATALITIES",
            "TOTAL_MAJOR_INJURIES",
            "TOTAL_MINOR_INJURIES",
            "SEVERITY_SCORE",
            "IS_SEVERE_CRASH",
            "IS_FATAL_CRASH",
            "MAJOR_OR_FATAL_COUNT"
        ]
    ].head()
)


Load cameras and humps

In [ ]:
cams = pd.read_csv(CAMERAS_PATH)
cams = cams.rename(columns={"CAMERA_LATITUDE": "LATITUDE","CAMERA_LONGITUDE": "LONGITUDE"})
cams = cams.dropna(subset=["LATITUDE", "LONGITUDE"]).copy()
cams["LATITUDE"]  = pd.to_numeric(cams["LATITUDE"], errors="coerce")
cams["LONGITUDE"] = pd.to_numeric(cams["LONGITUDE"], errors="coerce")
cams = cams.dropna(subset=["LATITUDE", "LONGITUDE"]).copy()

# Speed humps
humps = pd.read_csv(SPEEDHUMPS_PATH)
humps = humps.dropna(subset=["LATITUDE","LONGITUDE"]).copy()
humps["LATITUDE"]  = pd.to_numeric(humps["LATITUDE"], errors="coerce")
humps["LONGITUDE"] = pd.to_numeric(humps["LONGITUDE"], errors="coerce")
humps = humps.dropna(subset=["LATITUDE","LONGITUDE"]).copy()

Summaries of total number accidents, speed cameras, and speedbumps to help figure out what time of clustering should be done

In [ ]:
print("=== Current Totals (pre-distance) ===")
print(f"Crashes being counted: {len(df):,}")
print(f"Speed cameras: {len(cams):,}")
print(f"Speed humps:   {len(humps):,}")
print(f"Total devices: {len(cams) + len(humps):,}")

There are few crashes and total devices, so we can just do brute force distance calculations using haversine metrics to figure out distance between each crash and its nearest device. We can do this because there are only a total of 1,210 * 589 different calculations that are being run. I also used Euclidean distances as I did in the night accidents

Before we do that though, we must assign all speed bumps, cameras, and accidents onto their actual street so that when we do our distance calculations they are all on the proper road

Load the DDOT lines

In [ ]:
# --- 3.a) DDOT lines → standardized street name ("street_base") ---
import re, numpy as np, pandas as pd, geopandas as gpd
from shapely.geometry import Point

CRS_METERS = 32618
MAX_LATERAL_M = 20  # lateral tolerance to snap a point to a street

# Load and project
gdf_limits_4326 = gpd.read_file(SPEED_GEOJSON_PATH)
if gdf_limits_4326.crs is None:
    gdf_limits_4326 = gdf_limits_4326.set_crs(4326)
gdf_limits_m = gdf_limits_4326.to_crs(CRS_METERS).copy()

# Pick a DDOT street-name column and normalize it
name_cands_ddot = [c for c in ["STREETNAME","ROUTENAME","NAME","FULLNAME","SEGMENTNAME"] if c in gdf_limits_m.columns]
ddot_name_col = name_cands_ddot[0] if name_cands_ddot else None

def norm_street(s: str) -> str:
    if s is None or (isinstance(s,float) and np.isnan(s)): return ""
    s = str(s).upper().strip()
    # remove quadrant suffixes and stray punctuation
    s = re.sub(r"\b(NE|NW|SE|SW)\b", "", s)
    s = re.sub(r"[^A-Z0-9 &'-]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

if ddot_name_col:
    gdf_limits_m["street_base"] = gdf_limits_m[ddot_name_col].apply(norm_street)
else:
    # no name column? fallback: empty (we can still use nearest line later if needed)
    gdf_limits_m["street_base"] = ""

# Keep geometry + street_base only (index will serve as a line id if needed later)
ddot_lines = gdf_limits_m[["geometry","street_base"]].copy()


Assign the devices to the nearest DDOT line

In [ ]:
# 2) ASSIGN DEVICES — map to nearest DDOT line, inherit street_base
# Expects: cams, humps dataframes with LONGITUDE/LATITUDE

# Build device GeoDataFrames (WGS84 → meters)
gdf_cams_4326  = gpd.GeoDataFrame(cams.copy(),  geometry=gpd.points_from_xy(cams["LONGITUDE"],  cams["LATITUDE"]),  crs=4326)
gdf_humps_4326 = gpd.GeoDataFrame(humps.copy(), geometry=gpd.points_from_xy(humps["LONGITUDE"], humps["LATITUDE"]), crs=4326)
gdf_cams_4326["device_kind"]  = "camera"
gdf_humps_4326["device_kind"] = "hump"

cams_m    = gdf_cams_4326.to_crs(CRS_METERS)
humps_m   = gdf_humps_4326.to_crs(CRS_METERS)
devices_m = pd.concat([cams_m[["geometry","device_kind"]],
                       humps_m[["geometry","device_kind"]]], ignore_index=True)

# Snap to nearest DDOT line; inherit street_base
dev2line = gpd.sjoin_nearest(devices_m, ddot_lines, how="left", distance_col="dev_dist_to_line_m") \
             .rename(columns={"index_right":"ddot_idx"}) \
             .reset_index().rename(columns={"index":"device_index"})

dev2line["dev_valid_line"] = dev2line["dev_dist_to_line_m"] <= MAX_LATERAL_M
dev2line["street_base"]    = dev2line["street_base"].fillna("").astype(str)

print("Devices valid to street:", int(dev2line["dev_valid_line"].sum()), "/", len(dev2line))


Assign crashes to their nearest DDOT line

In [ ]:
# 3) ASSIGN CRASHES — map to nearest DDOT line, inherit street_base
# Expects: df (filtered crashes) with LONGITUDE/LATITUDE

gdf_crashes_4326 = gpd.GeoDataFrame(df.copy(), geometry=gpd.points_from_xy(df["LONGITUDE"], df["LATITUDE"]), crs=4326)
crashes_m = gdf_crashes_4326.to_crs(CRS_METERS)

cr2line = gpd.sjoin_nearest(crashes_m, ddot_lines, how="left", distance_col="cr_dist_to_line_m") \
           .rename(columns={"index_right":"ddot_idx"}) \
           .reset_index().rename(columns={"index":"crash_index"})

cr2line["cr_valid_line"] = cr2line["cr_dist_to_line_m"] <= MAX_LATERAL_M
cr2line["street_base"]   = cr2line["street_base"].fillna("").astype(str)

print("Crashes valid to street:", int(cr2line["cr_valid_line"].sum()), "/", len(cr2line))


Drop any device or crashes not assigned to ddot line

In [ ]:
# 3.5) DROP INVALID — keep only records with a valid street assignment
before_dev, before_cr = len(dev2line), len(cr2line)

dev_on_street = dev2line[dev2line["dev_valid_line"] & dev2line["street_base"].ne("")].copy()
cr_on_street  = cr2line[cr2line["cr_valid_line"] & cr2line["street_base"].ne("")].copy()

print("Devices kept:", len(dev_on_street), "/", before_dev,
      "| Crashes kept:", len(cr_on_street), "/", before_cr)


Now caculate the euclidean distances

In [ ]:
# 4) DISTANCES — nearest device on the SAME STREET (keep INF if none)

import numpy as np
import geopandas as gpd
import pandas as pd

# Start from devices_m (has geometry + device_kind) and add street_base from dev_on_street
devices_m_idxed = devices_m.reset_index().rename(columns={"index": "device_index"})  # geometry, device_kind
devices_m_idxed = devices_m_idxed.merge(
    dev_on_street[["device_index", "street_base"]],
    on="device_index",
    how="left"
)

rows = []
for street, cr_grp in cr_on_street.groupby("street_base"):
    dev_grp = devices_m_idxed[devices_m_idxed["street_base"] == street]

    if dev_grp.empty:
        tmp = cr_grp.copy()
        tmp["dist_to_device_same_street_m"] = np.inf
        tmp["nearest_device_index"] = pd.NA
        tmp["nearest_device_kind"]  = pd.NA
        rows.append(tmp)
        continue

    dev_gdf = gpd.GeoDataFrame(
        dev_grp[["geometry", "device_index", "device_kind"]].copy(),
        geometry="geometry",
        crs=CRS_METERS
    )

    tmp = gpd.sjoin_nearest(
        cr_grp.set_geometry("geometry"),
        dev_gdf,
        how="left",
        distance_col="dist_to_device_same_street_m"
    ).rename(columns={"index_right": "_dev_join_row"}).copy()

    tmp["nearest_device_index"] = tmp["device_index"]
    tmp["nearest_device_kind"]  = tmp["device_kind"]
    rows.append(tmp)

same_street = pd.concat(rows, ignore_index=True) if rows else cr_on_street.copy()

finite = np.isfinite(same_street["dist_to_device_same_street_m"])
print("Same-street nearest-device distances computed.")
print(f"Finite distances: {int(finite.sum())} of {len(same_street)}")
print(
    same_street[["crash_index", "street_base", "dist_to_device_same_street_m"]]
    .head(8)
    .to_string(index=False)
)


compute the avg block size from ddot

In [ ]:
# 5) THRESHOLD — compute from DDOT segment lengths (block size)

import numpy as np
import geopandas as gpd

# Ensure DDOT lines exist and are in meters CRS
try:
    _ = gdf_limits_m
except NameError:
    gdf_limits_4326 = gpd.read_file(SPEED_GEOJSON_PATH)
    if gdf_limits_4326.crs is None:
        gdf_limits_4326 = gdf_limits_4326.set_crs(4326)
    gdf_limits_m = gdf_limits_4326.to_crs(CRS_METERS).copy()

# Compute each DDOT segment's length (meters)
gdf_limits_m["seg_len_m"] = gdf_limits_m.geometry.length

# Mean and median block sizes
mean_len   = float(gdf_limits_m["seg_len_m"].mean())
median_len = float(gdf_limits_m["seg_len_m"].median())

# ✅ Use the MEDIAN block length as the threshold
THRESHOLD_M = 59.4

print(f"Block length — mean: {mean_len:,.1f} m | median: {median_len:,.1f} m")
print(f"Using THRESHOLD_M = {THRESHOLD_M:,.1f} m (median block size)")


Remove the crashes that are inside of the threshold so that clustering can be ready

In [ ]:
# 6) FILTER — label within/outside by THRESHOLD_M, but KEEP outside/inf for clustering

import numpy as np
import pandas as pd

# Expect same_street from Cell 4
is_finite   = np.isfinite(same_street["dist_to_device_same_street_m"])
within_mask = is_finite & (same_street["dist_to_device_same_street_m"] <= THRESHOLD_M)

kept_within_threshold     = same_street[within_mask].copy()
remaining_for_clustering  = same_street[~within_mask].copy()   # includes finite>THRESHOLD_M and inf

within_pct  = (len(kept_within_threshold) / len(same_street)) * 100
outside_pct = 100 - within_pct

print(f"Within threshold: {len(kept_within_threshold):,} / {len(same_street):,} "
      f"({within_pct:.2f}%)")
print(f"Outside threshold: {len(remaining_for_clustering):,} / {len(same_street):,} "
      f"({outside_pct:.2f}%)")

finite_outside = remaining_for_clustering[np.isfinite(remaining_for_clustering["dist_to_device_same_street_m"])]
infinite_rows  = remaining_for_clustering[~np.isfinite(remaining_for_clustering["dist_to_device_same_street_m"])]

print(f"Remaining for clustering: {len(remaining_for_clustering):,} "
      f"(finite>{THRESHOLD_M:.1f} m: {len(finite_outside):,}; inf: {len(infinite_rows):,})")

# Minimal inputs for clustering (needs the DDOT segment id)
cluster_input = remaining_for_clustering[["crash_index","ddot_idx","geometry"]].copy()
cluster_input = cluster_input.rename(columns={"ddot_idx":"cr_ddot_idx"})


Clustering using complete linkage with the 59.4 meter threshold (the median size of a block in dc)

In [ ]:
# 7) CLUSTER — complete linkage per DDOT segment using THRESHOLD_M as cutoff

import numpy as np
from scipy.cluster.hierarchy import linkage, fcluster

# Use the same median-based threshold from Cell 5
CLUSTER_CUTOFF_M = THRESHOLD_M

# ✅ Reset index so positions are 0..N-1 (prevents IndexError)
cluster_input = cluster_input.reset_index(drop=True)

labels = np.full(len(cluster_input), -1, dtype=int)

for seg_id, grp in cluster_input.groupby("cr_ddot_idx"):
    idx = grp.index.to_numpy()  # now 0..N-1 safe
    coords = np.c_[grp.geometry.x.values, grp.geometry.y.values]

    n = len(coords)
    if n == 1:
        labels[idx] = 0
        continue
    if n == 2:
        d = np.linalg.norm(coords[0] - coords[1])
        if d <= CLUSTER_CUTOFF_M:
            labels[idx] = 0
        else:
            labels[idx[0]] = 0
            labels[idx[1]] = 1
        continue

    # Hierarchical (complete-linkage) clustering in Euclidean meters
    Z = linkage(coords, method="complete", metric="euclidean")
    lbl = fcluster(Z, t=CLUSTER_CUTOFF_M, criterion="distance") - 1  # zero-based
    labels[idx] = lbl

cluster_input["cluster_id_complete"] = labels

print(f"Complete-linkage clustering done. Cutoff = {CLUSTER_CUTOFF_M:.1f} m (median block size)")
print(cluster_input[["cr_ddot_idx","cluster_id_complete"]].head(10).to_string(index=False))



Build variables for each cluster (still in crs)

In [ ]:
# =========================
# Cluster Severity Table
# (RANK, N_CRASHES, SEVERITY_SUM, AVG_SEVERITY,
#  SEVERE_CRASH_PERCENTAGE, FATAL_CRASH_PERCENTAGE, AVG_LON, AVG_LAT)
# =========================

import numpy as np
import pandas as pd
import geopandas as gpd
from IPython.display import display

# 1) Ensure one row per crash in cluster_input
clust_unique = cluster_input.drop_duplicates("crash_index").copy()

# 2) Build one-row-per-crash severity table FROM THE ORIGINAL CRASH DATAFRAME
#    Do NOT recompute from same_street.
#    Assumes df already has these columns correctly computed:
#    SEVERITY_SCORE, IS_SEVERE_CRASH, IS_FATAL_CRASH,
#    TOTAL_FATALITIES, TOTAL_MAJOR_INJURIES, TOTAL_MINOR_INJURIES

sev_per_crash = (
    df.reset_index()
      .rename(columns={"index": "crash_index"})
      [[
          "crash_index",
          "SEVERITY_SCORE",
          "CAPPED_SEVERITY",
          "IS_SEVERE_CRASH",
          "IS_FATAL_CRASH",
          "TOTAL_FATALITIES",
          "TOTAL_MAJOR_INJURIES",
          "TOTAL_MINOR_INJURIES"
      ]]
      .copy()
)

# Safety: force numeric
fill_zero_cols = [
    "SEVERITY_SCORE",
    "CAPPED_SEVERITY",
    "IS_SEVERE_CRASH",
    "IS_FATAL_CRASH",
    "TOTAL_FATALITIES",
    "TOTAL_MAJOR_INJURIES",
    "TOTAL_MINOR_INJURIES"
]
for c in fill_zero_cols:
    sev_per_crash[c] = pd.to_numeric(sev_per_crash[c], errors="coerce").fillna(0)

# Optional but smart: ensure one row per crash_index in severity table
sev_per_crash = sev_per_crash.drop_duplicates("crash_index")

# 3) Join crash-level severity metrics to unique cluster points
ci = clust_unique.merge(sev_per_crash, on="crash_index", how="left")

# Fill any missing joined values with 0
for c in fill_zero_cols:
    ci[c] = pd.to_numeric(ci[c], errors="coerce").fillna(0)

# 4) Aggregate by cluster and compute centroids in meters
ci["x"] = ci.geometry.x
ci["y"] = ci.geometry.y

cluster_stats = (
    ci.groupby(["cr_ddot_idx", "cluster_id_complete"], as_index=False)
      .agg(
          N_CRASHES=("crash_index", "size"),
          SEVERITY_SUM=("SEVERITY_SCORE", "sum"),
          CAPPED_SEVERITY_SUM=("CAPPED_SEVERITY", "sum"),
          SEVERE_CRASH_COUNT=("IS_SEVERE_CRASH", "sum"),
          FATAL_CRASH_COUNT=("IS_FATAL_CRASH", "sum"),
          TOTAL_FATALITIES=("TOTAL_FATALITIES", "sum"),
          TOTAL_MAJOR_INJURIES=("TOTAL_MAJOR_INJURIES", "sum"),
          TOTAL_MINOR_INJURIES=("TOTAL_MINOR_INJURIES", "sum"),
          avg_x=("x", "mean"),
          avg_y=("y", "mean")
      )
)

# 5) Add reviewer-requested metrics
cluster_stats["AVG_SEVERITY"] = (
    cluster_stats["SEVERITY_SUM"] / cluster_stats["N_CRASHES"]
)

cluster_stats["AVG_CAPPED_SEVERITY"] = (
    cluster_stats["CAPPED_SEVERITY_SUM"] / cluster_stats["N_CRASHES"]
)

cluster_stats["SEVERE_CRASH_PERCENTAGE"] = (
    100 * cluster_stats["SEVERE_CRASH_COUNT"] / cluster_stats["N_CRASHES"]
)

cluster_stats["FATAL_CRASH_PERCENTAGE"] = (
    100 * cluster_stats["FATAL_CRASH_COUNT"] / cluster_stats["N_CRASHES"]
)

cluster_stats["FATALITIES_PER_CRASH"] = (
    cluster_stats["TOTAL_FATALITIES"] / cluster_stats["N_CRASHES"]
)

AADT Normalization by Exposure

Make cluster points

In [ ]:
# AADT step 1: build one point per cluster centroid (still in CRS_METERS)

cluster_stats = cluster_stats.copy()

cluster_pts = gpd.GeoDataFrame(
    cluster_stats,
    geometry=gpd.points_from_xy(cluster_stats["avg_x"], cluster_stats["avg_y"]),
    crs=CRS_METERS
)

print(cluster_pts.shape)
display(cluster_pts[[
    "N_CRASHES",
    "SEVERITY_SUM",
    "CAPPED_SEVERITY_SUM",
    "avg_x",
    "avg_y",
    "geometry"
]].head())

Load AADT

In [ ]:
import os, zipfile

zip_path = "/content/drive/MyDrive/Traffic_Volume_-_2024 (3).zip"
extract_path = "/content/drive/MyDrive/aadt"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_path)

print(os.listdir(extract_path))

In [ ]:
AADT_PATH = "/content/drive/MyDrive/aadt/Traffic_Volume_-_2024.shp"

In [ ]:
aadt = gpd.read_file(AADT_PATH)

if aadt.crs is None:
    raise ValueError("AADT shapefile has no CRS metadata.")

aadt = aadt.to_crs(CRS_METERS).copy()

print(aadt.shape)
print(aadt.crs)
print(aadt.columns.tolist())

Match each cluster centroid to exactly one nearest AADT segment and compute both raw and capped risk metrics

In [ ]:
# Tie-safe nearest AADT match: keep exactly one segment per cluster centroid

AADT_COL = "AADT"

aadt_small = aadt[["geometry", AADT_COL, "ROUTEID"]].copy()
aadt_small[AADT_COL] = pd.to_numeric(aadt_small[AADT_COL], errors="coerce")
aadt_small = aadt_small[aadt_small[AADT_COL].notna() & (aadt_small[AADT_COL] > 0)].copy()
aadt_small = aadt_small.reset_index(drop=True)

# preserve a stable cluster key for deduping after sjoin_nearest
cluster_pts = cluster_pts.reset_index(drop=True).copy()
cluster_pts["CLUSTER_KEY"] = cluster_pts.index

cluster_aadt_raw = gpd.sjoin_nearest(
    cluster_pts,
    aadt_small,
    how="left",
    distance_col="DIST_TO_AADT_M"
).rename(columns={
    AADT_COL: "MATCHED_AADT",
    "ROUTEID": "MATCHED_ROUTEID"
})

print("Rows before join:", len(cluster_pts))
print("Rows after raw nearest join:", len(cluster_aadt_raw))

# If ties occur, keep exactly one row per cluster:
# 1) shortest distance
# 2) if still tied, larger AADT
# 3) if still tied, stable ROUTEID sort
cluster_aadt = (
    cluster_aadt_raw
    .sort_values(
        by=["CLUSTER_KEY", "DIST_TO_AADT_M", "MATCHED_AADT", "MATCHED_ROUTEID"],
        ascending=[True, True, False, True]
    )
    .drop_duplicates(subset="CLUSTER_KEY", keep="first")
    .sort_values("CLUSTER_KEY")
    .reset_index(drop=True)
)

if len(cluster_aadt) != len(cluster_stats):
    raise ValueError("Still not one row per cluster after tie-breaking.")

cluster_aadt["EXPOSURE_4YR"] = cluster_aadt["MATCHED_AADT"] * 365 * 5
cluster_aadt["RISK"] = cluster_aadt["SEVERITY_SUM"] / cluster_aadt["EXPOSURE_4YR"]
cluster_aadt["CAPPED_RISK"] = (
    cluster_aadt["CAPPED_SEVERITY_SUM"] / cluster_aadt["EXPOSURE_4YR"]
)

# clean downstream table
cluster_stats = pd.DataFrame(
    cluster_aadt.drop(columns=["geometry", "index_right", "CLUSTER_KEY"], errors="ignore")
)



In [ ]:
cluster_stats["RISK_PER_100M"] = cluster_stats["RISK"] * 100_000_000
cluster_stats["CAPPED_RISK_PER_100M"] = cluster_stats["CAPPED_RISK"] * 100_000_000

Display as ranked by severity sum

Convert to lon/lat and create table

In [ ]:
# 7) Convert centroids to lon/lat for mapping
centers_g = gpd.GeoDataFrame(
    cluster_stats,
    geometry=gpd.points_from_xy(cluster_stats["avg_x"], cluster_stats["avg_y"]),
    crs=CRS_METERS
).to_crs(4326)

cluster_df = (
    centers_g
    .assign(
        AVG_LON=lambda d: d.geometry.x,
        AVG_LAT=lambda d: d.geometry.y
    )
    .drop(columns=["geometry", "cr_ddot_idx", "cluster_id_complete", "avg_x", "avg_y"])
)

# 8) Keep requested columns
c# 8) Keep requested columns + AADT/risk columns
cluster_df = cluster_df[
    [
        "N_CRASHES",
        "SEVERITY_SUM",
        "CAPPED_SEVERITY_SUM",
        "AVG_SEVERITY",
        "AVG_CAPPED_SEVERITY",
        "SEVERE_CRASH_PERCENTAGE",
        "FATAL_CRASH_PERCENTAGE",
        "MATCHED_AADT",
        "DIST_TO_AADT_M",
        "MATCHED_ROUTEID",
        "EXPOSURE_4YR",
        "RISK",
        "CAPPED_RISK",
        "RISK_PER_100M",
        "CAPPED_RISK_PER_100M",
        "AVG_LON",
        "AVG_LAT"
    ]
]

# 9) Sort by total burden first, then severity intensity, then frequency
cluster_df = cluster_df.sort_values(
    ["SEVERITY_SUM", "AVG_SEVERITY", "N_CRASHES"],
    ascending=[False, False, False]
).reset_index(drop=True)

# 10) Add rank column at front
cluster_df.insert(0, "RANK", np.arange(1, len(cluster_df) + 1))

print("Cluster Table:")
display(cluster_df.head(10))



Publication refined table

In [ ]:
# Create separate publication-ready results table
results_table = cluster_df[
    [
        "N_CRASHES",
        "SEVERITY_SUM",
        "CAPPED_SEVERITY_SUM",
        "AVG_SEVERITY",
        "SEVERE_CRASH_PERCENTAGE",
        "FATAL_CRASH_PERCENTAGE",
        "AVG_LON",
        "AVG_LAT"
    ]
].copy()

# Show top 10
print("Results Table:")
display(results_table.head(10))

capped severity table

In [ ]:
# =========================
# Capped-Severity Ranked Cluster Table
# Keeps original rank alongside new capped rank
# =========================

cluster_df_capped = cluster_df.copy()

# Preserve original ranking from the original table
cluster_df_capped = cluster_df_capped.rename(columns={"RANK": "ORIGINAL_RANK"})

# Re-sort by capped severity burden first, then capped severity intensity, then frequency
cluster_df_capped = cluster_df_capped.sort_values(
    ["CAPPED_SEVERITY_SUM", "AVG_CAPPED_SEVERITY", "N_CRASHES"],
    ascending=[False, False, False]
).reset_index(drop=True)

# Add new capped ranking at the front
cluster_df_capped.insert(0, "CAPPED_RANK", np.arange(1, len(cluster_df_capped) + 1))

print("Capped-Severity Ranked Cluster Table:")
display(cluster_df_capped.head(10))

Condensed Capped Severity Sum Table

In [ ]:
# Create separate publication-ready results table
results_table = cluster_df[
    [
        "N_CRASHES",
        "SEVERITY_SUM",
        "CAPPED_SEVERITY_SUM",
        "AVG_SEVERITY",
        "SEVERE_CRASH_PERCENTAGE",
        "FATAL_CRASH_PERCENTAGE",
        "AVG_LON",
        "AVG_LAT"
    ]
].copy()

# Sort by capped severity sum
results_table = results_table.sort_values(
    ["CAPPED_SEVERITY_SUM", "SEVERITY_SUM", "N_CRASHES"],
    ascending=[False, False, False]
).reset_index(drop=True)

# Show top 10
print("Results Table ranked by capped severity sum:")
display(results_table.head(10))

Spearman Analysis

In [ ]:
# =========================
# Spearman Rank Correlation
# =========================

from scipy.stats import spearmanr

# Make sure we're comparing aligned clusters
# (same rows, same clusters)
df_compare = cluster_df_capped.copy()

# Use ORIGINAL_RANK (from severity) and CAPPED_RANK
rho, pval = spearmanr(
    df_compare["ORIGINAL_RANK"],
    df_compare["CAPPED_RANK"]
)

print(f"Spearman correlation (rho): {rho:.4f}")
print(f"P-value: {pval:.4e}")

Top 10 Overlay

In [ ]:
# =========================
# Top-10 Overlap
# =========================

k = 10

df_compare = cluster_df_capped.copy()

# Top 10 by original ranking
top_original = set(
    df_compare.nsmallest(k, "ORIGINAL_RANK").index
)

# Top 10 by capped ranking
top_capped = set(
    df_compare.nsmallest(k, "CAPPED_RANK").index
)

# Overlap
overlap = len(top_original & top_capped)
overlap_pct = overlap / k * 100

print(f"Top {k} overlap: {overlap} clusters")
print(f"Overlap percentage: {overlap_pct:.1f}%")

In [ ]:
# =========================
# Frequency (N_CRASHES) Ranked Cluster Table
# Includes severity and capped ranks
# =========================

cluster_df_freq = cluster_df_capped.copy()

# Rename for clarity (optional but cleaner)
cluster_df_freq = cluster_df_freq.rename(columns={
    "ORIGINAL_RANK": "SEVERITY_RANK",
    "CAPPED_RANK": "CAPPED_SEVERITY_RANK"
})

# Sort by frequency
cluster_df_freq = cluster_df_freq.sort_values(
    ["N_CRASHES", "SEVERITY_SUM", "AVG_SEVERITY"],
    ascending=[False, False, False]
).reset_index(drop=True)

# Add frequency rank at the front
cluster_df_freq.insert(0, "FREQUENCY_RANK", np.arange(1, len(cluster_df_freq) + 1))

print("Frequency-Ranked Cluster Table:")
display(cluster_df_freq.head(10))

Frequency vs Severity and Frequency vs Capped Severity

In [ ]:
# =========================
# Spearman: Frequency vs Others
# =========================

from scipy.stats import spearmanr

df_compare = cluster_df_freq.copy()

# Severity vs Frequency
rho_sf, p_sf = spearmanr(
    df_compare["SEVERITY_RANK"],
    df_compare["FREQUENCY_RANK"]
)

# Capped vs Frequency
rho_cf, p_cf = spearmanr(
    df_compare["CAPPED_SEVERITY_RANK"],
    df_compare["FREQUENCY_RANK"]
)

print("Spearman Correlations (Frequency Comparisons):\n")

print(f"Severity vs Frequency:  rho = {rho_sf:.4f}, p = {p_sf:.2e}")
print(f"Capped vs Frequency:   rho = {rho_cf:.4f}, p = {p_cf:.2e}")

Top 10 overlay for Frequency vs Severity and Frequency vs Capped Severity

In [ ]:
# =========================
# Top-10 Overlap: Frequency vs Others
# =========================

k = 10
df_compare = cluster_df_freq.copy()

# ---- Severity vs Frequency ----
top_severity = set(df_compare.nsmallest(k, "SEVERITY_RANK").index)
top_frequency = set(df_compare.nsmallest(k, "FREQUENCY_RANK").index)

overlap_sf = len(top_severity & top_frequency)
overlap_sf_pct = overlap_sf / k * 100

# ---- Capped vs Frequency ----
top_capped = set(df_compare.nsmallest(k, "CAPPED_SEVERITY_RANK").index)

overlap_cf = len(top_capped & top_frequency)
overlap_cf_pct = overlap_cf / k * 100

# ---- Print results ----
print(f"Top {k} Overlap:\n")

print(f"Severity vs Frequency:  {overlap_sf}/{k} ({overlap_sf_pct:.1f}%)")
print(f"Capped vs Frequency:   {overlap_cf}/{k} ({overlap_cf_pct:.1f}%)")

RISK ANALYSIS

ranked by Severity Sum Risk

In [ ]:
table_severity_risk = cluster_df.sort_values(
    ["RISK_PER_100M", "SEVERITY_SUM", "N_CRASHES"],
    ascending=[False, False, False]
).reset_index(drop=True)

table_severity_risk.insert(0, "RISK_RANK", np.arange(1, len(table_severity_risk) + 1))
table_severity_risk["RANK_CHANGE"] = table_severity_risk["RANK"] - table_severity_risk["RISK_RANK"]

display(table_severity_risk[[
    "RISK_RANK",
    "RANK",
    "RANK_CHANGE",
    "N_CRASHES",
    "SEVERITY_SUM",
    "CAPPED_SEVERITY_SUM",
    "MATCHED_AADT",
    "DIST_TO_AADT_M",
    "RISK_PER_100M",
    "CAPPED_RISK_PER_100M",
    "AVG_LON",
    "AVG_LAT"
]].head(10))

Refined Table

In [ ]:
# =========================
# SPEEDING AADT STEP 4B
# Clean display table — separate from full risk table
# =========================

speeding_risk_clean_table = table_severity_risk[[
    "RANK_CHANGE",
    "N_CRASHES",
    "SEVERITY_SUM",
    "RISK_PER_100M",
    "AVG_LON",
    "AVG_LAT"
]].copy()

print("Speeding clean severity-risk table:")
display(speeding_risk_clean_table.head(10))

Saving all the threshold tables

In [ ]:
# save full ranked table for this threshold
if abs(THRESHOLD_M - 44.55) < 1e-9:
    cluster_44_55 = table_severity_risk.copy()
    cluster_44_55["THRESHOLD_M"] = THRESHOLD_M
    cluster_44_55.to_csv(f"{SAVE_DIR}/cluster_44_55.csv", index=False)

elif abs(THRESHOLD_M - 59.4) < 1e-9:
    cluster_59_4 = table_severity_risk.copy()
    cluster_59_4["THRESHOLD_M"] = THRESHOLD_M
    cluster_59_4.to_csv(f"{SAVE_DIR}/cluster_59_4.csv", index=False)

elif abs(THRESHOLD_M - 74.25) < 1e-9:
    cluster_74_25 = table_severity_risk.copy()
    cluster_74_25["THRESHOLD_M"] = THRESHOLD_M
    cluster_74_25.to_csv(f"{SAVE_DIR}/cluster_74_25.csv", index=False)

elif abs(THRESHOLD_M - 150) < 1e-9:
    cluster_150 = table_severity_risk.copy()
    cluster_150["THRESHOLD_M"] = THRESHOLD_M
    cluster_150.to_csv(f"{SAVE_DIR}/cluster_150.csv", index=False)

elif abs(THRESHOLD_M - 200) < 1e-9:
    cluster_200 = table_severity_risk.copy()
    cluster_200["THRESHOLD_M"] = THRESHOLD_M
    cluster_200.to_csv(f"{SAVE_DIR}/cluster_200.csv", index=False)

elif abs(THRESHOLD_M - 250) < 1e-9:
    cluster_250 = table_severity_risk.copy()
    cluster_250["THRESHOLD_M"] = THRESHOLD_M
    cluster_250.to_csv(f"{SAVE_DIR}/cluster_250.csv", index=False)

print(f"Saved full ranked cluster table for threshold = {THRESHOLD_M} m")
print(f"Rows in saved table: {len(table_severity_risk):,}")

Ranked by Capped Severity Sum risk

In [ ]:
table_capped_risk = cluster_df.sort_values(
    ["CAPPED_RISK_PER_100M", "CAPPED_SEVERITY_SUM", "N_CRASHES"],
    ascending=[False, False, False]
).reset_index(drop=True)

table_capped_risk.insert(0, "CAPPED_RISK_RANK", np.arange(1, len(table_capped_risk) + 1))
table_capped_risk["RANK_CHANGE"] = table_capped_risk["RANK"] - table_capped_risk["CAPPED_RISK_RANK"]

display(table_capped_risk[[
    "CAPPED_RISK_RANK",
    "RANK",
    "RANK_CHANGE",
    "N_CRASHES",
    "SEVERITY_SUM",
    "CAPPED_SEVERITY_SUM",
    "MATCHED_AADT",
    "DIST_TO_AADT_M",
    "RISK_PER_100M",
    "CAPPED_RISK_PER_100M",
    "AVG_LON",
    "AVG_LAT"
]].head(10))

Spearman Severity Sum Risk vs Severity Sum

In [ ]:
from scipy.stats import spearmanr

rho1, pval1 = spearmanr(
    table_severity_risk["RANK"],          # original severity rank
    table_severity_risk["RISK_RANK"]      # risk rank
)

print("Severity vs Risk")
print("Spearman rho:", rho1)
print("p-value:", pval1)

Top 10 Overlay - Severity Sum Risk vs Severity Sum

In [ ]:
top10_severity = set(
    zip(cluster_df.head(10)["AVG_LON"], cluster_df.head(10)["AVG_LAT"])
)

top10_risk = set(
    zip(table_severity_risk.head(10)["AVG_LON"], table_severity_risk.head(10)["AVG_LAT"])
)

overlap_sr = len(top10_severity & top10_risk)
print("Severity vs Risk overlap:", overlap_sr, "/ 10")

Spearman Capped Severity Sum Risk vs Capped Severity Sum

In [ ]:
from scipy.stats import spearmanr

cs = cluster_stats.copy()

cs["CAPPED_SEVERITY_RANK"] = cs["CAPPED_SEVERITY_SUM"].rank(method="first", ascending=False)
cs["CAPPED_RISK_RANK"] = cs["CAPPED_RISK_PER_100M"].rank(method="first", ascending=False)

rho2, p2 = spearmanr(
    cs["CAPPED_SEVERITY_RANK"],
    cs["CAPPED_RISK_RANK"]
)

print("Capped vs Capped Risk:", rho2, p2)

Top 10 Overlay Capped Severity Sum Risk vs Capped Severity Sum

In [ ]:
top10_capped = set(
    zip(
        cluster_df.sort_values("CAPPED_SEVERITY_SUM", ascending=False)
        .head(10)["AVG_LON"],
        cluster_df.sort_values("CAPPED_SEVERITY_SUM", ascending=False)
        .head(10)["AVG_LAT"]
    )
)

top10_capped_risk = set(
    zip(
        table_capped_risk.head(10)["AVG_LON"],
        table_capped_risk.head(10)["AVG_LAT"]
    )
)

overlap_cr = len(top10_capped & top10_capped_risk)
print("Capped vs Capped Risk overlap:", overlap_cr, "/ 10")

map

In [ ]:
40# === MAP: All speeding crashes (uniform style) + all devices + TOP 10 clusters (size+color by SEVERITY_SUM) ===
import numpy as np
import pandas as pd
import geopandas as gpd
import folium
from branca.colormap import linear

# 0) Ensure devices exist in WGS84
try:
    _ = gdf_cams_4326
    _ = gdf_humps_4326
except NameError:
    gdf_cams_4326  = gpd.GeoDataFrame(
        cams.copy(),
        geometry=gpd.points_from_xy(cams["LONGITUDE"], cams["LATITUDE"]),
        crs=4326
    )
    gdf_humps_4326 = gpd.GeoDataFrame(
        humps.copy(),
        geometry=gpd.points_from_xy(humps["LONGITUDE"], humps["LATITUDE"]),
        crs=4326
    )

# 1) Convert crashes to WGS84
crashes_4326 = gpd.GeoDataFrame(same_street.copy(), geometry="geometry", crs=CRS_METERS).to_crs(4326)

# 2) Top 10 clusters by SEVERITY_SUM (highest)
if "cluster_df" in globals() and isinstance(cluster_df, pd.DataFrame) and not cluster_df.empty:
    top10_clusters = cluster_df.sort_values("SEVERITY_SUM", ascending=False).head(10).copy()
else:
    top10_clusters = pd.DataFrame()

# 3) Cluster styling (variable color + radius by SEVERITY_SUM like your sample)
if not top10_clusters.empty:
    vmin = float(top10_clusters["SEVERITY_SUM"].min())
    vmax = float(top10_clusters["SEVERITY_SUM"].max())
    if vmin == vmax:
        vmin = 0.0
    cmap = linear.Reds_09.scale(vmin, vmax)

    def radius_by_sev(sev):
        sev = float(sev)
        return 6 if vmax == vmin else 6 + 20 * (sev - vmin) / (vmax - vmin)
else:
    cmap = None
    def radius_by_sev(sev):  # unused
        return 10

# 4) Build the map
m = folium.Map(location=[38.9072, -77.0369], zoom_start=12, tiles=None, max_zoom=19)
folium.TileLayer('cartodbpositron', name='OSM (Carto Positron)', control=True).add_to(m)
folium.TileLayer(
    tiles="https://services.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Esri, Maxar, Earthstar Geographics, and the GIS User Community",
    name="ESRI World Street (z≤19)", max_zoom=19, control=True
).add_to(m)

# 5) Layer: All devices (cameras & humps)
fg_cams  = folium.FeatureGroup(name="Devices – Speed Cameras", show=False)
for _, r in gdf_cams_4326.iterrows():
    folium.CircleMarker(
        location=[float(r.geometry.y), float(r.geometry.x)],
        radius=3, color="#1976d2",
        fill=True, fill_opacity=0.9,
        popup=folium.Popup("Speed Camera", max_width=200)
    ).add_to(fg_cams)
fg_cams.add_to(m)

fg_humps = folium.FeatureGroup(name="Devices – Speed Humps", show=False)
for _, r in gdf_humps_4326.iterrows():
    folium.CircleMarker(
        location=[float(r.geometry.y), float(r.geometry.x)],
        radius=3, color="#2e7d32",
        fill=True, fill_opacity=0.9,
        popup=folium.Popup("Speed Hump", max_width=200)
    ).add_to(fg_humps)
fg_humps.add_to(m)

# 6) Layer: All speeding crashes (UNIFORM style: same size + same color)
fg_crashes = folium.FeatureGroup(name="All Speeding Crashes (uniform)", show=True)
for _, r in crashes_4326.iterrows():
    dt = r["FROMDATE"].date() if "FROMDATE" in r and pd.notna(r["FROMDATE"]) else ""
    sev = int(r.get("SEVERITY_SCORE", 0)) if "SEVERITY_SCORE" in r else ""

    pop = folium.Popup(
        f"Date: {dt}<br>Severity score: {sev}",
        max_width=280
    )

    folium.CircleMarker(
        location=[float(r.geometry.y), float(r.geometry.x)],
        radius=2.5,
        color="#0d6efd",
        fill=True,
        fill_color="",
        fill_opacity=0.65,
        weight=1,
        popup=pop
    ).add_to(fg_crashes)
fg_crashes.add_to(m)

# 7) Layer: TOP 10 cluster centroids (variable style by SEVERITY_SUM)
if not top10_clusters.empty:
    fg_clusters = folium.FeatureGroup(name="Top 10 Clusters (by SEVERITY_SUM)", show=True)

    for rank, (_, row) in enumerate(top10_clusters.iterrows(), start=1):
        lat = float(row["AVG_LAT"])
        lon = float(row["AVG_LON"])
        sev = float(row["SEVERITY_SUM"])
        n   = int(row["N_CRASHES"])

        popup = folium.Popup(
            f"<b>Rank:</b> {rank}<br>"
            f"<b>Crashes:</b> {n}<br>"
            f"<b>Severity sum:</b> {int(sev)}<br>"
            f"<b>Center:</b> {lat:.6f}, {lon:.6f}",
            max_width=360
        )

        folium.CircleMarker(
            location=[lat, lon],
            radius=radius_by_sev(sev),
            color=cmap(sev),
            fill=True,
            fill_color=cmap(sev),
            fill_opacity=0.9,
            weight=2,
            popup=popup,
            tooltip=f"Cluster Rank {rank} | Sev {int(sev)} | Cr {n}"
        ).add_to(fg_clusters)

    fg_clusters.add_to(m)

    cmap.caption = "Cluster severity (SEVERITY_SUM) — Top 10 only"
    cmap.add_to(m)
else:
    print("[Note] cluster_df not found or empty — skipping top-10 cluster layer.")

folium.LayerControl(collapsed=False).add_to(m)
display(m)



Reload saved tables

In [ ]:
import pandas as pd

SAVE_DIR = "/content/drive/MyDrive/cluster_threshold_tables"

cluster_44_55 = pd.read_csv(f"{SAVE_DIR}/cluster_44_55.csv")
cluster_59_4  = pd.read_csv(f"{SAVE_DIR}/cluster_59_4.csv")
cluster_74_25 = pd.read_csv(f"{SAVE_DIR}/cluster_74_25.csv")

cluster_150   = pd.read_csv(f"{SAVE_DIR}/cluster_150.csv")
cluster_200   = pd.read_csv(f"{SAVE_DIR}/cluster_200.csv")
cluster_250   = pd.read_csv(f"{SAVE_DIR}/cluster_250.csv")

print("Loaded all saved cluster tables.")

In [ ]:
import os

# =========================================================
# OUTPUT FOLDER
# =========================================================
out_dir = "/content/drive/MyDrive/crashmap_outputs/final_tables"
os.makedirs(out_dir, exist_ok=True)

# =========================================================
# BASE TABLE (contains everything already)
# =========================================================
base_df = cluster_df.copy()

# =========================================================
# FUNCTION TO CREATE AND SAVE A RANKED TABLE
# =========================================================
def save_ranked_table(df, sort_cols, name):
    ranked = (
        df.sort_values(sort_cols, ascending=[False]*len(sort_cols))
          .reset_index(drop=True)
          .copy()
    )

    # 🔥 FIX: remove existing RANK if it exists
    if "RANK" in ranked.columns:
        ranked = ranked.drop(columns=["RANK"])

    ranked.insert(0, "RANK", range(1, len(ranked)+1))

    path = os.path.join(out_dir, name)
    ranked.to_csv(path, index=False)

    print(f"Saved: {path}")

# =========================================================
# SAVE ALL 4 METRICS
# =========================================================

# 1) Severity
save_ranked_table(
    base_df,
    ["SEVERITY_SUM", "AVG_SEVERITY", "N_CRASHES"],
    f"clusters_{THRESHOLD_M}_severity_full.csv"
)

# 2) Capped Severity
save_ranked_table(
    base_df,
    ["CAPPED_SEVERITY_SUM", "AVG_CAPPED_SEVERITY", "N_CRASHES"],
    f"clusters_{THRESHOLD_M}_capped_severity_full.csv"
)

# 3) Risk
save_ranked_table(
    base_df,
    ["RISK", "SEVERITY_SUM", "N_CRASHES"],
    f"clusters_{THRESHOLD_M}_risk_full.csv"
)

# 4) Capped Risk
save_ranked_table(
    base_df,
    ["CAPPED_RISK", "CAPPED_SEVERITY_SUM", "N_CRASHES"],
    f"clusters_{THRESHOLD_M}_capped_risk_full.csv"
)

In [ ]:
base_path = "/content/drive/MyDrive/crashmap_outputs/final_tables"

In [ ]:
import pandas as pd
import os

base_path = "/content/drive/MyDrive/crashmap_outputs/final_tables"

def load_by_keyword(keyword):
    for f in os.listdir(base_path):
        if keyword in f:
            print(f"Loaded: {f}")
            return pd.read_csv(os.path.join(base_path, f))
    raise FileNotFoundError(f"No file found for keyword: {keyword}")

# =========================
# 59.4 m
# =========================
sev_59   = load_by_keyword("59.4_severity")
cap_59   = load_by_keyword("59.4_capped_severity")
risk_59  = load_by_keyword("59.4_risk")
crisk_59 = load_by_keyword("59.4_capped_risk")

# =========================
# 200 m
# =========================
sev_200   = load_by_keyword("200_severity")
cap_200   = load_by_keyword("200_capped_severity")
risk_200  = load_by_keyword("200_risk")
crisk_200 = load_by_keyword("200_capped_risk")

Convert all tables into Geopandas to check different thresholds

In [ ]:
import geopandas as gpd

def make_gdf(df):
    gdf = gpd.GeoDataFrame(
        df.copy(),
        geometry=gpd.points_from_xy(df["AVG_LON"], df["AVG_LAT"]),
        crs="EPSG:4326"
    )
    return gdf.to_crs(epsg=32618)   # meters

gdf_44 = make_gdf(cluster_44_55)
gdf_59 = make_gdf(cluster_59_4)
gdf_74 = make_gdf(cluster_74_25)

gdf_150 = make_gdf(cluster_150)
gdf_200 = make_gdf(cluster_200)
gdf_250 = make_gdf(cluster_250)

print("GeoDataFrames created.")

Matching for 59.4 to 44.55 in both directions including reverses

In [ ]:
# =========================
# 59.4 vs 44.55 (both directions)
# =========================

CORRIDOR_DISTANCE_M = 100

# --- 59.4 → 44.55 ---
match_59_to_44 = gpd.sjoin_nearest(
    gdf_59,
    gdf_44[["RISK_RANK", "geometry"]],
    how="left",
    distance_col="MATCH_DIST_M",
    lsuffix="59",
    rsuffix="44"
)

match_59_to_44["IS_MATCH"] = match_59_to_44["MATCH_DIST_M"] <= CORRIDOR_DISTANCE_M

n_total_59 = len(match_59_to_44)
n_match_59 = match_59_to_44["IS_MATCH"].sum()
pct_59 = n_match_59 / n_total_59


# --- 44.55 → 59.4 ---
match_44_to_59 = gpd.sjoin_nearest(
    gdf_44,
    gdf_59[["RISK_RANK", "geometry"]],
    how="left",
    distance_col="MATCH_DIST_M",
    lsuffix="44",
    rsuffix="59"
)

match_44_to_59["IS_MATCH"] = match_44_to_59["MATCH_DIST_M"] <= CORRIDOR_DISTANCE_M

n_total_44 = len(match_44_to_59)
n_match_44 = match_44_to_59["IS_MATCH"].sum()
pct_44 = n_match_44 / n_total_44


# --- Results ---
print("59.4 → 44.55")
print(f"Matched: {n_match_59} / {n_total_59}")
print(f"Overlap: {pct_59:.4f}")

print("\n44.55 → 59.4")
print(f"Matched: {n_match_44} / {n_total_44}")
print(f"Overlap: {pct_44:.4f}")

filter matches

In [ ]:
matched_59_44 = match_59_to_44[match_59_to_44["IS_MATCH"]].copy()

print(len(matched_59_44))

Spearman (44.55 vs 59.4)

In [ ]:
from scipy.stats import spearmanr

rho, pval = spearmanr(
    matched_59_44["RISK_RANK_59"],
    matched_59_44["RISK_RANK_44"]
)

print(f"Spearman rho: {rho:.4f}")
print(f"p-value: {pval:.4e}")

59.4 to 74.25 in both directions including reverses

In [ ]:
# =========================
# 59.4 vs 74.25 (both directions)
# =========================

CORRIDOR_DISTANCE_M = 100

# --- 59.4 → 74.25 ---
match_59_to_74 = gpd.sjoin_nearest(
    gdf_59,
    gdf_74[["RISK_RANK", "geometry"]],
    how="left",
    distance_col="MATCH_DIST_M",
    lsuffix="59",
    rsuffix="74"
)

match_59_to_74["IS_MATCH"] = match_59_to_74["MATCH_DIST_M"] <= CORRIDOR_DISTANCE_M

n_total_59 = len(match_59_to_74)
n_match_59 = match_59_to_74["IS_MATCH"].sum()
pct_59 = n_match_59 / n_total_59

# --- 74.25 → 59.4 ---
match_74_to_59 = gpd.sjoin_nearest(
    gdf_74,
    gdf_59[["RISK_RANK", "geometry"]],
    how="left",
    distance_col="MATCH_DIST_M",
    lsuffix="74",
    rsuffix="59"
)

match_74_to_59["IS_MATCH"] = match_74_to_59["MATCH_DIST_M"] <= CORRIDOR_DISTANCE_M

n_total_74 = len(match_74_to_59)
n_match_74 = match_74_to_59["IS_MATCH"].sum()
pct_74 = n_match_74 / n_total_74

print("59.4 → 74.25")
print(f"Matched: {n_match_59} / {n_total_59}")
print(f"Overlap: {pct_59:.4f}")

print("\n74.25 → 59.4")
print(f"Matched: {n_match_74} / {n_total_74}")
print(f"Overlap: {pct_74:.4f}")

Spearman

In [ ]:
from scipy.stats import spearmanr

CORRIDOR_DISTANCE_M = 100

match_59_to_74 = gpd.sjoin_nearest(
    gdf_59,
    gdf_74[["RISK_RANK", "geometry"]],
    how="left",
    distance_col="MATCH_DIST_M",
    lsuffix="59",
    rsuffix="74"
)

match_59_to_74["IS_MATCH"] = match_59_to_74["MATCH_DIST_M"] <= CORRIDOR_DISTANCE_M
matched = match_59_to_74[match_59_to_74["IS_MATCH"]]

rho, pval = spearmanr(matched["RISK_RANK_59"], matched["RISK_RANK_74"])

print("59.4 → 74.25")
print(f"Overlap: {len(matched)/len(match_59_to_74):.4f}")
print(f"Spearman rho: {rho:.4f}, p={pval:.2e}")

150 vs 200 both ways, including reverse (literature robustness)

In [ ]:
# =========================
# 200 vs 150 (both directions)
# =========================

CORRIDOR_DISTANCE_M = 100

# --- 200 → 150 ---
match_200_to_150 = gpd.sjoin_nearest(
    gdf_200,
    gdf_150[["RISK_RANK", "geometry"]],
    how="left",
    distance_col="MATCH_DIST_M",
    lsuffix="200",
    rsuffix="150"
)

match_200_to_150["IS_MATCH"] = match_200_to_150["MATCH_DIST_M"] <= CORRIDOR_DISTANCE_M

n_total_200 = len(match_200_to_150)
n_match_200 = match_200_to_150["IS_MATCH"].sum()
pct_200 = n_match_200 / n_total_200

# --- 150 → 200 ---
match_150_to_200 = gpd.sjoin_nearest(
    gdf_150,
    gdf_200[["RISK_RANK", "geometry"]],
    how="left",
    distance_col="MATCH_DIST_M",
    lsuffix="150",
    rsuffix="200"
)

match_150_to_200["IS_MATCH"] = match_150_to_200["MATCH_DIST_M"] <= CORRIDOR_DISTANCE_M

n_total_150 = len(match_150_to_200)
n_match_150 = match_150_to_200["IS_MATCH"].sum()
pct_150 = n_match_150 / n_total_150

print("200 → 150")
print(f"Matched: {n_match_200} / {n_total_200}")
print(f"Overlap: {pct_200:.4f}")

print("\n150 → 200")
print(f"Matched: {n_match_150} / {n_total_150}")
print(f"Overlap: {pct_150:.4f}")

Spearman

In [ ]:
from scipy.stats import spearmanr

CORRIDOR_DISTANCE_M = 100

match_200_to_150 = gpd.sjoin_nearest(
    gdf_200,
    gdf_150[["RISK_RANK", "geometry"]],
    how="left",
    distance_col="MATCH_DIST_M",
    lsuffix="200",
    rsuffix="150"
)

match_200_to_150["IS_MATCH"] = match_200_to_150["MATCH_DIST_M"] <= CORRIDOR_DISTANCE_M
matched = match_200_to_150[match_200_to_150["IS_MATCH"]]

rho, pval = spearmanr(matched["RISK_RANK_200"], matched["RISK_RANK_150"])

print("200 → 150")
print(f"Overlap: {len(matched)/len(match_200_to_150):.4f}")
print(f"Spearman rho: {rho:.4f}, p={pval:.2e}")

200 vs 250 in both ways, including reverse (literature robustness)

In [ ]:
# =========================
# 200 vs 250 (both directions)
# =========================

CORRIDOR_DISTANCE_M = 100

# --- 200 → 250 ---
match_200_to_250 = gpd.sjoin_nearest(
    gdf_200,
    gdf_250[["RISK_RANK", "geometry"]],
    how="left",
    distance_col="MATCH_DIST_M",
    lsuffix="200",
    rsuffix="250"
)

match_200_to_250["IS_MATCH"] = match_200_to_250["MATCH_DIST_M"] <= CORRIDOR_DISTANCE_M

n_total_200 = len(match_200_to_250)
n_match_200 = match_200_to_250["IS_MATCH"].sum()
pct_200 = n_match_200 / n_total_200

# --- 250 → 200 ---
match_250_to_200 = gpd.sjoin_nearest(
    gdf_250,
    gdf_200[["RISK_RANK", "geometry"]],
    how="left",
    distance_col="MATCH_DIST_M",
    lsuffix="250",
    rsuffix="200"
)

match_250_to_200["IS_MATCH"] = match_250_to_200["MATCH_DIST_M"] <= CORRIDOR_DISTANCE_M

n_total_250 = len(match_250_to_200)
n_match_250 = match_250_to_200["IS_MATCH"].sum()
pct_250 = n_match_250 / n_total_250

print("200 → 250")
print(f"Matched: {n_match_200} / {n_total_200}")
print(f"Overlap: {pct_200:.4f}")

print("\n250 → 200")
print(f"Matched: {n_match_250} / {n_total_250}")
print(f"Overlap: {pct_250:.4f}")

Spearman

In [ ]:
from scipy.stats import spearmanr

CORRIDOR_DISTANCE_M = 100

match_200_to_250 = gpd.sjoin_nearest(
    gdf_200,
    gdf_250[["RISK_RANK", "geometry"]],
    how="left",
    distance_col="MATCH_DIST_M",
    lsuffix="200",
    rsuffix="250"
)

match_200_to_250["IS_MATCH"] = match_200_to_250["MATCH_DIST_M"] <= CORRIDOR_DISTANCE_M
matched = match_200_to_250[match_200_to_250["IS_MATCH"]]

rho, pval = spearmanr(matched["RISK_RANK_200"], matched["RISK_RANK_250"])

print("200 → 250")
print(f"Overlap: {len(matched)/len(match_200_to_250):.4f}")
print(f"Spearman rho: {rho:.4f}, p={pval:.2e}")

59.4 vs 200 in both directions, including reverse (cross check between block size and literature review)

In [ ]:
# =========================
# 59.4 vs 200 (both directions)
# =========================

CORRIDOR_DISTANCE_M = 100

# --- 59.4 → 200 ---
match_59_to_200 = gpd.sjoin_nearest(
    gdf_59,
    gdf_200[["RISK_RANK", "geometry"]],
    how="left",
    distance_col="MATCH_DIST_M",
    lsuffix="59",
    rsuffix="200"
)

match_59_to_200["IS_MATCH"] = match_59_to_200["MATCH_DIST_M"] <= CORRIDOR_DISTANCE_M

n_total_59 = len(match_59_to_200)
n_match_59 = match_59_to_200["IS_MATCH"].sum()
pct_59 = n_match_59 / n_total_59

# --- 200 → 59.4 ---
match_200_to_59 = gpd.sjoin_nearest(
    gdf_200,
    gdf_59[["RISK_RANK", "geometry"]],
    how="left",
    distance_col="MATCH_DIST_M",
    lsuffix="200",
    rsuffix="59"
)

match_200_to_59["IS_MATCH"] = match_200_to_59["MATCH_DIST_M"] <= CORRIDOR_DISTANCE_M

n_total_200 = len(match_200_to_59)
n_match_200 = match_200_to_59["IS_MATCH"].sum()
pct_200 = n_match_200 / n_total_200

print("59.4 → 200")
print(f"Matched: {n_match_59} / {n_total_59}")
print(f"Overlap: {pct_59:.4f}")

print("\n200 → 59.4")
print(f"Matched: {n_match_200} / {n_total_200}")
print(f"Overlap: {pct_200:.4f}")

Spearman

In [ ]:
from scipy.stats import spearmanr

CORRIDOR_DISTANCE_M = 100

match_59_to_200 = gpd.sjoin_nearest(
    gdf_59,
    gdf_200[["RISK_RANK", "geometry"]],
    how="left",
    distance_col="MATCH_DIST_M",
    lsuffix="59",
    rsuffix="200"
)

match_59_to_200["IS_MATCH"] = match_59_to_200["MATCH_DIST_M"] <= CORRIDOR_DISTANCE_M
matched = match_59_to_200[match_59_to_200["IS_MATCH"]]

rho, pval = spearmanr(matched["RISK_RANK_59"], matched["RISK_RANK_200"])

print("59.4 → 200")
print(f"Overlap: {len(matched)/len(match_59_to_200):.4f}")
print(f"Spearman rho: {rho:.4f}, p={pval:.2e}")

DBSCAN

In [ ]:
# =========================
# DBSCAN (STRICT VERSION)
# =========================

import numpy as np
import pandas as pd
import geopandas as gpd
from sklearn.cluster import DBSCAN

gdf_db = ci.copy()

assert "geometry" in gdf_db.columns
assert "SEVERITY_SCORE" in gdf_db.columns
assert "cr_ddot_idx" in gdf_db.columns

gdf_db["X"] = gdf_db.geometry.x
gdf_db["Y"] = gdf_db.geometry.y

EPS = THRESHOLD_M      # 59.4
MIN_SAMPLES = 2

labels = np.full(len(gdf_db), -1, dtype=int)
next_cluster_id = 0

for seg_id, grp in gdf_db.groupby("cr_ddot_idx"):
    idx = grp.index.to_numpy()
    coords = grp[["X", "Y"]].to_numpy()

    # if fewer than min_samples points on this road segment,
    # they cannot form a DBSCAN cluster
    if len(coords) < MIN_SAMPLES:
        labels[idx] = -1
        continue

    db = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES, metric="euclidean")
    local = db.fit_predict(coords)

    unique = sorted([c for c in np.unique(local) if c != -1])
    mapping = {c: next_cluster_id + i for i, c in enumerate(unique)}

    for i, loc_idx in enumerate(idx):
        labels[loc_idx] = mapping[local[i]] if local[i] != -1 else -1

    next_cluster_id += len(unique)

gdf_db["cluster_id_dbscan"] = labels

n_noise = int((gdf_db["cluster_id_dbscan"] == -1).sum())
n_clusters = gdf_db.loc[gdf_db["cluster_id_dbscan"] != -1, "cluster_id_dbscan"].nunique()

print(f"Noise: {n_noise} ({100*n_noise/len(gdf_db):.2f}%)")
print(f"Clusters formed: {n_clusters}")

db_pts = gdf_db[gdf_db["cluster_id_dbscan"] != -1].copy()

dbscan_table = (
    db_pts.groupby("cluster_id_dbscan")
    .agg(
        N_CRASHES=("cluster_id_dbscan", "size"),
        SEVERITY_SUM=("SEVERITY_SCORE", "sum"),
        AVG_X=("X", "mean"),
        AVG_Y=("Y", "mean"),
    )
    .reset_index()
)

dbscan_gdf = gpd.GeoDataFrame(
    dbscan_table,
    geometry=gpd.points_from_xy(dbscan_table["AVG_X"], dbscan_table["AVG_Y"]),
    crs=gdf_db.crs
).to_crs(4326)

dbscan_gdf["AVG_LON"] = dbscan_gdf.geometry.x
dbscan_gdf["AVG_LAT"] = dbscan_gdf.geometry.y

dbscan_top = (
    dbscan_gdf
    .sort_values(["SEVERITY_SUM", "N_CRASHES"], ascending=[False, False])
    .reset_index(drop=True)
)

dbscan_top["DBSCAN_RANK"] = np.arange(1, len(dbscan_top) + 1)

display(dbscan_top.head(10))

Comparring Complete Linkage vs DBSCAN

In [ ]:
import pandas as pd
from scipy.stats import spearmanr

# COMPLETE-LINKAGE TABLE
cl = cluster_df.copy()   # <-- adjust name if needed

cl = cl.sort_values(
    ["SEVERITY_SUM", "N_CRASHES"], ascending=[False, False]
).reset_index(drop=True)

cl["CL_RANK"] = range(1, len(cl)+1)

# DBSCAN TABLE (already ranked)
db = dbscan_top.copy()

# keep only needed columns
cl = cl[["SEVERITY_SUM", "AVG_LON", "AVG_LAT", "CL_RANK"]]
db = db[["cluster_id_dbscan", "SEVERITY_SUM", "AVG_LON", "AVG_LAT", "DBSCAN_RANK"]]

match clusters and do the overlay with DBSCAN

In [ ]:
import numpy as np
from sklearn.neighbors import BallTree

# ---- 1) Rank tables ----
cl = cluster_df.copy().sort_values(
    ["SEVERITY_SUM", "N_CRASHES"], ascending=[False, False]
).reset_index(drop=True)
cl["CL_RANK"] = range(1, len(cl) + 1)

db = dbscan_top.copy().sort_values(
    ["SEVERITY_SUM", "N_CRASHES"], ascending=[False, False]
).reset_index(drop=True)
db["DBSCAN_RANK"] = range(1, len(db) + 1)

# ---- 2) Convert lat/lon -> meters (equirectangular, good for city scale) ----
lat0 = np.deg2rad(pd.concat([cl["AVG_LAT"], db["AVG_LAT"]]).mean())
R = 6371000.0

def to_xy(lat, lon):
    lat = np.deg2rad(lat)
    lon = np.deg2rad(lon)
    x = R * lon * np.cos(lat0)
    y = R * lat
    return np.c_[x, y]

# ---- 3) Overlap at K (Euclidean in meters) ----
for k in [5, 10, 20]:
    cl_xy = to_xy(*cl.nsmallest(k, "CL_RANK")[["AVG_LAT","AVG_LON"]].T.values)
    db_xy = to_xy(*db.nsmallest(k, "DBSCAN_RANK")[["AVG_LAT","AVG_LON"]].T.values)

    tree = BallTree(db_xy, metric="euclidean")
    dist, _ = tree.query(cl_xy, k=1)

    matches = int((dist.flatten() <= 100).sum())
    print(f"Top-{k}: {matches}/{k} ({matches/k:.0%})")

DBSCAN vs Risk

In [ ]:
# =====================================================
# SPEEDING: DBSCAN vs AADT-NORMALIZED COMPLETE LINKAGE
# TRUE TOP-K OVERLAY ONLY
# =====================================================

import numpy as np
from sklearn.neighbors import BallTree

# ---- 1) Risk-ranked complete-linkage table ----
cl = table_severity_risk.copy().sort_values(
    ["RISK_PER_100M", "SEVERITY_SUM", "N_CRASHES"],
    ascending=[False, False, False]
).reset_index(drop=True)

cl["RISK_RANK"] = range(1, len(cl) + 1)

# ---- 2) Already-created DBSCAN table ----
db = dbscan_top.copy().sort_values(
    ["SEVERITY_SUM", "N_CRASHES"],
    ascending=[False, False]
).reset_index(drop=True)

db["DBSCAN_RANK"] = range(1, len(db) + 1)

# ---- 3) Convert lat/lon -> meters, matching your original speeding DBSCAN cell ----
lat0 = np.deg2rad(pd.concat([cl["AVG_LAT"], db["AVG_LAT"]]).mean())
R = 6371000.0

def to_xy(lat, lon):
    lat = np.deg2rad(lat)
    lon = np.deg2rad(lon)
    x = R * lon * np.cos(lat0)
    y = R * lat
    return np.c_[x, y]

# ---- 4) Top-k overlay ----
print("\nDBSCAN vs AADT-Normalized Complete Linkage")

for k in [5, 10, 20]:
    k_use = min(k, len(cl), len(db))

    cl_xy = to_xy(
        cl.nsmallest(k_use, "RISK_RANK")["AVG_LAT"],
        cl.nsmallest(k_use, "RISK_RANK")["AVG_LON"]
    )

    db_xy = to_xy(
        db.nsmallest(k_use, "DBSCAN_RANK")["AVG_LAT"],
        db.nsmallest(k_use, "DBSCAN_RANK")["AVG_LON"]
    )

    tree = BallTree(db_xy, metric="euclidean")
    dist, _ = tree.query(cl_xy, k=1)

    matches = int((dist.flatten() <= 100).sum())

    print(f"Top-{k_use}: {matches}/{k_use} ({matches/k_use:.0%})")

DDOT comparrison

In [ ]:
import geopandas as gpd

HIN_PATH = "/content/drive/MyDrive/High_Injury_Network.geojson"

hin = gpd.read_file(HIN_PATH)
print(hin.crs)
print(hin.columns.tolist())
print(hin.head())

Project HIN (DDOT CORRIDORS)

In [ ]:
import geopandas as gpd

CRS_METERS = 32618

# ensure CRS
if hin.crs is None:
    hin = hin.set_crs(4326)

# project to meters
hin = hin.to_crs(CRS_METERS)

# keep only geometry
hin = hin[["geometry"]].copy()

print("HIN ready:", len(hin))

Convert Tables to GeoPandas

In [ ]:
import geopandas as gpd

def to_gdf(df):
    return gpd.GeoDataFrame(
        df.copy(),
        geometry=gpd.points_from_xy(df["AVG_LON"], df["AVG_LAT"]),
        crs=4326
    ).to_crs(CRS_METERS)

attatch distance to HIN

In [ ]:
def attach_hin(df):
    pts = to_gdf(df)

    joined = gpd.sjoin_nearest(
        pts,
        hin,
        how="left",
        distance_col="DIST_TO_HIN_M"
    ).copy()

    # ensure one row per cluster (important)
    joined = (
        joined.sort_values(["RANK", "DIST_TO_HIN_M"])
              .drop_duplicates(subset="RANK")
              .reset_index(drop=True)
    )

    joined["WITHIN_59_4M"] = joined["DIST_TO_HIN_M"] <= 59.4
    joined["WITHIN_100M"]  = joined["DIST_TO_HIN_M"] <= 100
    joined["WITHIN_200M"]  = joined["DIST_TO_HIN_M"] <= 200

    return joined

apply to 4 datasets

In [ ]:
risk_59_hin  = attach_hin(risk_59)
sev_59_hin   = attach_hin(sev_59)

risk_200_hin = attach_hin(risk_200)
sev_200_hin  = attach_hin(sev_200)

main table

In [ ]:
import numpy as np
import pandas as pd

def summarize(df, threshold, metric):
    top5  = df[df["RANK"] <= 5]
    top10 = df[df["RANK"] <= 10]
    top20 = df[df["RANK"] <= 20]

    return {
        "CLUSTER": threshold,
        "METRIC": metric,

        "TOP5_100M": int(top5["WITHIN_100M"].sum()),
        "TOP10_100M": int(top10["WITHIN_100M"].sum()),
        "TOP20_100M": int(top20["WITHIN_100M"].sum()),

        "MEDIAN_DIST_M": round(df["DIST_TO_HIN_M"].median(), 1)
    }

table1 = pd.DataFrame([
    summarize(risk_59_hin,  "59.4 m", "Risk"),
    summarize(sev_59_hin,   "59.4 m", "Severity"),
    summarize(risk_200_hin, "200 m",  "Risk"),
    summarize(sev_200_hin,  "200 m",  "Severity"),
])

print("MAIN TABLE")
display(table1)

Robustness Test

In [ ]:
def summarize_robust(df, threshold, metric):
    top10 = df[df["RANK"] <= 10]

    return {
        "CLUSTER": threshold,
        "METRIC": metric,

        "TOP10_59_4M": int(top10["WITHIN_59_4M"].sum()),
        "TOP10_100M": int(top10["WITHIN_100M"].sum()),
        "TOP10_200M": int(top10["WITHIN_200M"].sum()),
    }

table2 = pd.DataFrame([
    summarize_robust(risk_59_hin,  "59.4 m", "Risk"),
    summarize_robust(sev_59_hin,   "59.4 m", "Severity"),
    summarize_robust(risk_200_hin, "200 m",  "Risk"),
    summarize_robust(sev_200_hin,  "200 m",  "Severity"),
])

print("ROBUSTNESS TABLE")
display(table2)